# Forward & Reverse Diffusion Processes
**Lecture — Politecnico di Milano**

This notebook provides a visual and mathematical deep-dive into the two processes at the heart of DDPM (Ho et al., 2020):

1. **Forward process** $q$ — gradually corrupts a data sample $\mathbf{x}_0$ into pure noise $\mathbf{x}_T$.
2. **Reverse process** $p_\theta$ — a neural network learns to undo the corruption, step by step.

By default, the notebook generates a synthetic image. **Set `IMAGE_PATH`** in the configuration cell to use your own image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
from PIL import Image
import os, warnings

warnings.filterwarnings('ignore')

# Ensures CUDA operations are executed synchronously (easier debugging)
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')

np.random.seed(0)
torch.manual_seed(0)
print('Libraries ready.')

In [ ]:
# ============================================================
# CONFIGURATION  ←  modify here
# ============================================================
IMAGE_PATH = None    # e.g. 'my_photo.jpg'  |  None → synthetic
IMAGE_SIZE = 64      # resize to this square
T_STEPS    = 1000    # total diffusion timesteps
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ── Image utilities ──────────────────────────────────────────────────────────
def make_synthetic_image(size=64):
    img = np.zeros((size, size, 3), dtype=np.float32)
    for i in range(size):
        img[i, :, 0] = i / size
        img[:, i, 1] = i / size
    img[:, :, 2] = 0.35
    cx = cy = size // 2
    r = size // 5
    Y, X = np.ogrid[:size, :size]
    img[(X-cx)**2 + (Y-cy)**2 < r**2] = [1.0, 0.9, 0.1]
    img[4:14, 4:14] = [1.0, 1.0, 1.0]
    return np.clip(img, 0, 1)

def load_image(path, size):
    if path and os.path.isfile(path):
        img = Image.open(path).convert('RGB').resize((size, size))
        arr = np.array(img, dtype=np.float32) / 255.0
    else:
        if path:
            print(f'[Warning] {path} not found — using synthetic image.')
        arr = make_synthetic_image(size)
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)  # (1,C,H,W)
    return tensor * 2.0 - 1.0   # [0,1] → [-1,1]

def tensor_to_display(t):
    '''Convert (C,H,W) tensor in [-1,1] to (H,W,C) numpy in [0,1]'''
    return ((t.cpu().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).numpy()

x0 = load_image(IMAGE_PATH, IMAGE_SIZE).to(DEVICE)
print(f'Image tensor shape (B, C, H, W): {tuple(x0.shape)}')

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(tensor_to_display(x0[0]))
ax.set_title('$\\mathbf{x}_0$ — Original image', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 1. Noise Schedules

The schedule $\{\beta_t\}_{t=1}^T$ controls **how fast** the signal is destroyed.

Define:
$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$$

$\bar{\alpha}_t$ is the **fraction of original signal** remaining at step $t$.

**Linear schedule** (Ho et al. 2020): $\beta_t$ grows linearly from $\beta_1=10^{-4}$ to $\beta_T=0.02$.  
**Cosine schedule** (Nichol & Dhariwal 2021): designed so $\bar{\alpha}_t$ follows a cosine curve — less aggressive early, preventing premature information loss.

In [ ]:
import math

def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

def cosine_beta_schedule(T, s=0.008):
    steps = torch.arange(T + 1, dtype=torch.float64)
    f = torch.cos(((steps / T) + s) / (1 + s) * math.pi / 2) ** 2
    alphas_cumprod = f / f[0]
    betas = 1.0 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clamp(betas, 1e-5, 0.9999).float()

def get_schedule(betas):
    alphas            = 1.0 - betas
    a_bar             = torch.cumprod(alphas, dim=0)
    sqrt_a_bar        = torch.sqrt(a_bar)
    sqrt_one_minus    = torch.sqrt(1.0 - a_bar)
    return dict(betas=betas, alphas=alphas, a_bar=a_bar,
                sqrt_a_bar=sqrt_a_bar, sqrt_one_minus=sqrt_one_minus)

T = T_STEPS
sched_lin = get_schedule(linear_beta_schedule(T).to(DEVICE))
sched_cos = get_schedule(cosine_beta_schedule(T).to(DEVICE))

t_axis = np.arange(T)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].plot(t_axis, sched_lin['betas'].cpu(), label='Linear',  color='steelblue')
axes[0].plot(t_axis, sched_cos['betas'].cpu(), label='Cosine',  color='tomato')
axes[0].set_title('$\\beta_t$ (noise variance added per step)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Timestep $t$'); axes[0].legend()

axes[1].plot(t_axis, sched_lin['a_bar'].cpu(), label='Linear', color='steelblue')
axes[1].plot(t_axis, sched_cos['a_bar'].cpu(), label='Cosine', color='tomato')
axes[1].axhline(0.5, color='gray', linestyle=':', linewidth=1.2, label='50% signal')
axes[1].set_title('$\\bar{\\alpha}_t$ (fraction of original signal)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Timestep $t$'); axes[1].legend()

axes[2].plot(t_axis, sched_lin['sqrt_a_bar'].cpu(),     color='steelblue', label='Lin $\\sqrt{\\bar\\alpha_t}$ (signal)')
axes[2].plot(t_axis, sched_lin['sqrt_one_minus'].cpu(), color='steelblue', linestyle='--', label='Lin $\\sqrt{1-\\bar\\alpha_t}$ (noise)')
axes[2].plot(t_axis, sched_cos['sqrt_a_bar'].cpu(),     color='tomato', label='Cos $\\sqrt{\\bar\\alpha_t}$ (signal)')
axes[2].plot(t_axis, sched_cos['sqrt_one_minus'].cpu(), color='tomato', linestyle='--', label='Cos $\\sqrt{1-\\bar\\alpha_t}$ (noise)')
axes[2].set_title('Signal / noise mixing coefficients', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Timestep $t$'); axes[2].legend(fontsize=7)

plt.suptitle('Noise Schedules: Linear vs Cosine', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 2. Forward Diffusion Process  $q(\mathbf{x}_t \mid \mathbf{x}_0)$

The forward process adds Gaussian noise step by step:
$$q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}\!\left(\mathbf{x}_t;\, \sqrt{1-\beta_t}\,\mathbf{x}_{t-1},\; \beta_t\,\mathbf{I}\right)$$

**Reparameterization trick** — we can sample at any $t$ directly from $\mathbf{x}_0$:
$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}\!\left(\mathbf{x}_t;\, \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1-\bar{\alpha}_t)\,\mathbf{I}\right)$$

Sampling in closed form:
$$\boxed{\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\varepsilon}}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

This is crucial for **training efficiency**: we never need to chain $t$ sequential steps.

In [ ]:
# We'll use the cosine schedule for the rest of the notebook
sched = sched_cos

def q_sample(x0, t_idx, sched, noise=None):
    '''Sample x_t ~ q(x_t | x_0) via the reparameterization trick.
    x0   : (B, C, H, W)
    t_idx: int or (B,) tensor
    Returns (x_t, noise)
    '''
    if noise is None:
        noise = torch.randn_like(x0)
    if isinstance(t_idx, int):
        t_idx = torch.tensor([t_idx], device=x0.device)
    s = sched['sqrt_a_bar'][t_idx].view(-1, 1, 1, 1)
    n = sched['sqrt_one_minus'][t_idx].view(-1, 1, 1, 1)
    return s * x0 + n * noise, noise

# Visualise forward process at selected timesteps
timesteps = [0, 50, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]

fixed_noise = torch.randn_like(x0)   # use the SAME noise instance for all t
noisy_images = []
for t in timesteps:
    xt, _ = q_sample(x0, t, sched, noise=fixed_noise)
    noisy_images.append(xt[0].cpu())

fig, axes = plt.subplots(2, len(timesteps), figsize=(22, 5))
for col, (t, xt) in enumerate(zip(timesteps, noisy_images)):
    snr = sched['sqrt_a_bar'][t].item() / (sched['sqrt_one_minus'][t].item() + 1e-8)
    axes[0, col].imshow(tensor_to_display(xt))
    axes[0, col].set_title(f't={t}\nSNR={snr:.2f}', fontsize=8)
    axes[0, col].axis('off')

    # Show the noise component
    noise_vis = ((fixed_noise[0].cpu().clamp(-3, 3) + 3) / 6).permute(1, 2, 0).numpy()
    noise_scaled = sched['sqrt_one_minus'][t].item() * fixed_noise[0].cpu()
    axes[1, col].imshow(tensor_to_display(noise_scaled))
    axes[1, col].set_title(f'$\\sqrt{{1-\\bar\\alpha_t}}\\cdot\\varepsilon$', fontsize=7)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Noisy image $x_t$', fontsize=9, rotation=90, labelpad=5)
axes[1, 0].set_ylabel('Noise component', fontsize=9, rotation=90, labelpad=5)
plt.suptitle('Forward Process $q(x_t|x_0)$ — cosine schedule (left=clean, right=pure noise)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── SNR analysis ─────────────────────────────────────────────────────────────
def snr_db(sched):
    s = sched['sqrt_a_bar'].cpu().numpy()
    n = sched['sqrt_one_minus'].cpu().numpy()
    return 20 * np.log10(s / (n + 1e-10))

t_axis = np.arange(T)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_axis, snr_db(sched_lin), color='steelblue', label='Linear schedule')
ax.plot(t_axis, snr_db(sched_cos), color='tomato',    label='Cosine schedule')
ax.axhline(0, color='black', linestyle='--', linewidth=1.2, label='SNR = 0 dB (equal signal & noise)')
ax.fill_between(t_axis, snr_db(sched_cos), 0,
                where=(snr_db(sched_cos) > 0), alpha=0.12, color='tomato', label='Signal dominant')
ax.fill_between(t_axis, snr_db(sched_cos), 0,
                where=(snr_db(sched_cos) < 0), alpha=0.12, color='purple', label='Noise dominant')
ax.set_title('Signal-to-Noise Ratio during Forward Process', fontsize=12, fontweight='bold')
ax.set_xlabel('Timestep $t$'); ax.set_ylabel('SNR (dB)')
ax.legend(fontsize=9); ax.grid(True)
plt.tight_layout(); plt.show()
print(f'SNR reaches 0 dB at approx t = {np.argmin(np.abs(snr_db(sched_lin))):d} (linear)')
print(f'SNR reaches 0 dB at approx t = {np.argmin(np.abs(snr_db(sched_cos))):d} (cosine)')

---
## 3. Reverse Diffusion Process for DDPM $p_\theta(\mathbf{x}_{t-1} \mid \mathbf{x}_t)$

The reverse process runs backwards: starting from $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$, iteratively denoise.

The **true posterior** $q(\mathbf{x}_{t-1}\mid \mathbf{x}_t, \mathbf{x}_0)$ is tractable:
$$q(\mathbf{x}_{t-1}\mid \mathbf{x}_t, \mathbf{x}_0) = \mathcal{N}(\mathbf{x}_{t-1};\; \tilde{\boldsymbol{\mu}}_t,\; \tilde{\beta}_t\,\mathbf{I})$$

with:
$$\tilde{\boldsymbol{\mu}}_t = \frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_t}{1-\bar{\alpha}_t}\,\mathbf{x}_0 + \frac{\sqrt{\alpha_t}\,(1-\bar{\alpha}_{t-1})}{1-\bar{\alpha}_t}\,\mathbf{x}_t, \qquad \tilde{\beta}_t = \frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}\,\beta_t$$

In practice, the network predicts noise $\boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, t)$, and we compute:
$$\boldsymbol{\mu}_\theta = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\boldsymbol{\varepsilon}_\theta\right)$$

**Oracle denoiser:** below we use the *true* stored noise as $\boldsymbol{\varepsilon}_\theta$ to show what a perfect denoiser would do.

In [ ]:
# Precompute posterior variance and other quantities needed for reverse
import torch.nn.functional as F

betas    = sched['betas']
alphas   = sched['alphas']
a_bar    = sched['a_bar']
a_bar_prev = F.pad(a_bar[:-1], (1, 0), value=1.0)    # a_bar at t-1

posterior_variance = betas * (1.0 - a_bar_prev) / (1.0 - a_bar)
posterior_variance = posterior_variance.clamp(min=1e-20)

def ddpm_reverse_step_oracle(x_t, t_idx, true_noise, sched, post_var):
    '''One DDPM reverse step using the TRUE noise (oracle).
    x_t        : (B, C, H, W) current noisy image
    t_idx      : int  current timestep
    true_noise : (B, C, H, W)  the actual epsilon used in the forward pass
    '''
    alpha_t   = sched['alphas'][t_idx]
    a_bar_t   = sched['a_bar'][t_idx]
    beta_t    = sched['betas'][t_idx]

    coeff  = beta_t / torch.sqrt(1.0 - a_bar_t)
    mean   = (1.0 / torch.sqrt(alpha_t)) * (x_t - coeff * true_noise)

    if t_idx > 0:
        z   = torch.randn_like(x_t)
        std = torch.sqrt(post_var[t_idx])
        return mean + std * z
    return mean

print('DDPM function defined.')

In [ ]:
# ── Oracle reverse trajectory ────────────────────────────────────────────────
# Start from x_T and reverse back to x_0 using the stored noise
print('Running oracle DDPM reverse process...')
xt = q_sample(x0, T_STEPS - 1, sched, noise=fixed_noise)[0].clone()

# choose ~12 evenly spaced reverse timesteps including 0 and T-1
stride = max(1, (T_STEPS - 1) // 11)
vis_steps = list(range(T_STEPS - 1, -1, -stride))
vis_steps = sorted(set(vis_steps + [0]), reverse=True)

reverse_images_ddpm = {}
for t_idx in range(T_STEPS - 1, -1, -1):
    if t_idx in vis_steps:
        reverse_images_ddpm[t_idx] = xt[0].cpu().clone()
    xt = ddpm_reverse_step_oracle(xt, t_idx, fixed_noise, sched, posterior_variance)
reverse_images_ddpm[0] = xt[0].cpu().clone()

steps_show = sorted(reverse_images_ddpm.keys(), reverse=True)
fig, axes = plt.subplots(1, len(steps_show), figsize=(22, 3))
for col, t in enumerate(steps_show):
    axes[col].imshow(tensor_to_display(reverse_images_ddpm[t]))
    axes[col].set_title(f't={t}', fontsize=9)
    axes[col].axis('off')
plt.suptitle('Reverse Process (Oracle: uses true noise) — $\\mathbf{x}_T$ → $\\hat{\\mathbf{x}}_0$', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 3. Reverse Diffusion Process for DDIM  $p_\theta(\mathbf{x}_{t-1} \mid \mathbf{x}_t)$

DDIM reformulates the reverse process as a **deterministic non-Markovian trajectory**.  
Starting from $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$, the model progressively reconstructs $\mathbf{x}_0$ without necessarily injecting additional random noise at each step.

Unlike DDPM, DDIM does **not** sample from a Gaussian posterior distribution during reverse diffusion.  
Instead, it first predicts the clean sample $\hat{\mathbf{x}}_0$ from the current noisy observation $\mathbf{x}_t$.

The forward diffusion marginal remains identical to DDPM:
$$
q(\mathbf{x}_t \mid \mathbf{x}_0)
=
\mathcal{N}
\left(
\sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\;
(1-\bar{\alpha}_t)\mathbf{I}
\right)
$$

Using the predicted noise $\boldsymbol{\varepsilon}_\theta(\mathbf{x}_t,t)$, the clean sample estimate is:
$$
\hat{\mathbf{x}}_0
=
\frac{
\mathbf{x}_t
-
\sqrt{1-\bar{\alpha}_t}\,
\boldsymbol{\varepsilon}_\theta
}{
\sqrt{\bar{\alpha}_t}
}
$$

The DDIM reverse update is then:
$$
\mathbf{x}_{t-1}
=
\sqrt{\bar{\alpha}_{t-1}}\,
\hat{\mathbf{x}}_0
+
\sqrt{1-\bar{\alpha}_{t-1}-\sigma_t^2}\,
\boldsymbol{\varepsilon}_\theta
+
\sigma_t\,\mathbf{z}
$$

where:
$$
\mathbf{z}\sim\mathcal{N}(\mathbf{0},\mathbf{I})
$$

and:
$$
\sigma_t
=
\eta
\sqrt{
\frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}
}
\sqrt{
1-\frac{\bar{\alpha}_t}{\bar{\alpha}_{t-1}}
}
$$

The parameter $\eta$ controls the stochasticity of the reverse process:

- $\eta = 1$  $\rightarrow$ DDIM behaves similarly to DDPM
- $\eta = 0$  $\rightarrow$ fully deterministic DDIM sampling

When $\eta = 0$, the update simplifies to:
$$
\mathbf{x}_{t-1}
=
\sqrt{\bar{\alpha}_{t-1}}\,
\hat{\mathbf{x}}_0
+
\sqrt{1-\bar{\alpha}_{t-1}}\,
\boldsymbol{\varepsilon}_\theta
$$

This deterministic formulation enables:

- accelerated sampling with fewer reverse steps,
- smoother latent interpolations,
- approximate inversion of the diffusion trajectory.

**Oracle denoiser:** below we use the *true* stored noise as $\boldsymbol{\varepsilon}_\theta$ to visualize the ideal DDIM reverse trajectory obtained with a perfect denoiser.

In [ ]:
def ddim_reverse_step_oracle(x_t, t_idx, true_noise, sched, eta=0.0):
    """
    One DDIM reverse step using the TRUE forward noise.

    Parameters
    ----------
    x_t : tensor
        Current noisy sample at timestep t

    t_idx : int
        Current timestep

    true_noise : tensor
        Ground-truth epsilon used in forward diffusion

    sched : dict
        Diffusion schedule dictionary

    eta : float
        Controls stochasticity:
            eta = 0   -> deterministic DDIM
            eta > 0   -> stochastic DDIM
    """

    a_bar_t = sched['a_bar'][t_idx]

    # previous cumulative alpha
    if t_idx > 0:
        a_bar_prev = sched['a_bar'][t_idx - 1]
    else:
        a_bar_prev = torch.tensor(1.0, device=x_t.device)

    # --------------------------------------------------------
    # Predict x0 using TRUE noise (oracle)
    # --------------------------------------------------------

    x0_pred = (x_t - torch.sqrt(1.0 - a_bar_t) * true_noise) / torch.sqrt(a_bar_t)

    # --------------------------------------------------------
    # DDIM sigma_t
    # --------------------------------------------------------

    sigma_t = (eta * torch.sqrt((1 - a_bar_prev) / (1 - a_bar_t)) * torch.sqrt(1 - a_bar_t / a_bar_prev))

    # --------------------------------------------------------
    # Direction pointing to x_t
    # --------------------------------------------------------

    dir_xt = torch.sqrt(torch.clamp(1.0 - a_bar_prev - sigma_t**2, min=0.0)) * true_noise

    # --------------------------------------------------------
    # Random noise (only if eta > 0)
    # --------------------------------------------------------

    if eta > 0 and t_idx > 0:
        z = torch.randn_like(x_t)
    else:
        z = torch.zeros_like(x_t)

    # --------------------------------------------------------
    # DDIM update
    # --------------------------------------------------------

    x_prev = (torch.sqrt(a_bar_prev) * x0_pred + dir_xt + sigma_t * z)

    return x_prev

print('DDIM function defined.')

In [ ]:
# ── Oracle reverse trajectory ────────────────────────────────────────────────
# Start from x_T and reverse back to x_0 using the stored noise
print('Running oracle reverse process...')
xt = q_sample(x0, T_STEPS - 1, sched, noise=fixed_noise)[0].clone()

# choose ~12 evenly spaced reverse timesteps including 0 and T-1
stride = max(1, (T_STEPS - 1) // 11)
vis_steps = list(range(T_STEPS - 1, -1, -stride))
vis_steps = sorted(set(vis_steps + [0]), reverse=True)

reverse_images_ddim = {}
for t_idx in range(T_STEPS - 1, -1, -1):
    if t_idx in vis_steps:
        reverse_images_ddim[t_idx] = xt[0].cpu().clone()
    xt = ddim_reverse_step_oracle(xt, t_idx, fixed_noise, sched, eta=0.0) # eta = 0.0: deterministic
reverse_images_ddim[0] = xt[0].cpu().clone()

steps_show = sorted(reverse_images_ddim.keys(), reverse=True)
fig, axes = plt.subplots(1, len(steps_show), figsize=(22, 3))
for col, t in enumerate(steps_show):
    axes[col].imshow(tensor_to_display(reverse_images_ddim[t]))
    axes[col].set_title(f't={t}', fontsize=9)
    axes[col].axis('off')
plt.suptitle('Reverse Process (Oracle: uses true noise) — $\\mathbf{x}_T$ → $\\hat{\\mathbf{x}}_0$', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Side-by-side: Forward (top) vs Reverse (bottom) ──────────────────────────
selected = [0, 100, 200, 300, 500, 700, 900, 999]
selected = vis_steps[::-1]
selected_reversed = selected[::-1]

fwd_imgs = []
for t in selected:
    xt_fwd, _ = q_sample(x0, t, sched, noise=fixed_noise)
    fwd_imgs.append(xt_fwd[0].cpu())
rev_show_ddpm = [reverse_images_ddpm[t] for t in selected_reversed if t in reverse_images_ddpm]
rev_show_ddim = [reverse_images_ddim[t] for t in selected_reversed if t in reverse_images_ddim]

fig = plt.figure(figsize=(22, 6))
gs  = gridspec.GridSpec(3, len(selected), height_ratios=[1, 1, 1], hspace=0.4)

for col in range(len(selected)):
    ax_top = fig.add_subplot(gs[0, col])
    ax_top.imshow(tensor_to_display(fwd_imgs[col]))
    ax_top.set_title(f't={selected[col]}', fontsize=9)
    ax_top.axis('off')

    ax_bot = fig.add_subplot(gs[1, col])
    ax_bot.imshow(tensor_to_display(rev_show_ddpm[col]))
    ax_bot.set_title(f't={selected_reversed[col]}', fontsize=9)
    ax_bot.axis('off')

    ax_bot = fig.add_subplot(gs[2, col])
    ax_bot.imshow(tensor_to_display(rev_show_ddim[col]))
    ax_bot.set_title(f't={selected_reversed[col]}', fontsize=9)
    ax_bot.axis('off')

fig.text(0.01, 0.78, 'Forward\n$q(x_t|x_0)$', va='center', ha='left', fontsize=11,
         color='steelblue', fontweight='bold')
fig.text(0.01, 0.50, 'Reverse DDPM \n$p_\\theta(x_{t-1}|x_t)$', va='center', ha='left', fontsize=11,
         color='tomato', fontweight='bold')
fig.text(0.01, 0.22, 'Reverse DDIM\n$p_\\theta(x_{t-1}|x_t)$', va='center', ha='left', fontsize=11,
         color='tomato', fontweight='bold')
fig.text(0.5, 0.95, 'Forward Process (top) vs Reverse Process (bottom)',
         ha='center', fontsize=14, fontweight='bold')
plt.show()

---
## Key Takeaways

| Concept | Formula | Intuition |
|---|---|---|
| Forward, one step | $q(x_t|x_{t-1}) = \mathcal{N}(\sqrt{1-\beta_t}\,x_{t-1},\, \beta_t I)$ | Slightly noisier at each step |
| Forward, closed form | $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ | **Can skip directly to any $t$** |
| Reverse posterior | $q(x_{t-1}|x_t, x_0) = \mathcal{N}(\tilde\mu_t, \tilde\beta_t I)$ | Tractable when $x_0$ known |
| Network learns | $\varepsilon_\theta(x_t, t) \approx \varepsilon$ | Predict the noise added |
| Training loss | $\mathcal{L} = \mathbb{E}[\|\varepsilon - \varepsilon_\theta(x_t, t)\|^2]$ | Simple MSE |

**Cosine vs Linear schedule:** cosine schedule preserves more signal early on (higher $\bar\alpha_t$ for small $t$), which helps train the denoiser on fine-grained details.